In [0]:
dbutils.widgets.text("catalog","dev")
catalog = dbutils.widgets.get("catalog")

### Cleaning customers raw data

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.table(f"{catalog}.bronze.bronze_customers")

In [0]:
# handle duplicate customer records, laeding/trailing spaces, inconsistantemail,missig values, invalid date
df_cleaned = df.withColumn("date_of_birth", F.to_date(F.col("date_of_birth"),"yyyy-MM-dd")) \
    .withColumn("registration_date", F.to_date(F.col("registration_date"),"yyyy-MM-dd")) \
        .withColumn("updated_at", F.to_date(F.col("updated_at"),"yyyy-MM-dd")) \
            .filter(F.col("email").contains('@') | (F.col("email").contains('.com'))) \
                .withColumn("email", F.lower(F.trim(F.col("email")))) \
                    .withColumn("customer_id", F.trim(F.col("customer_id"))) \
                        .withColumn("first_name", F.trim(F.col("first_name"))) \
                            .withColumn("last_name", F.trim(F.col("last_name"))) \
                                .withColumn("address", F.lower(F.trim(F.col("address")))) \
                                    .withColumn("city", F.lower(F.trim(F.col("city")))) \
                                        .withColumn("state", F.lower(F.trim(F.col("state")))) \
                                            .withColumn("postal_code", F.trim(F.col("postal_code"))) \
                                                .withColumn("customer_segment", F.lower(F.trim(F.col("customer_segment")))) \
                                                    .withColumn("customer_status", F.lower(F.trim(F.col("customer_status")))) \
                                                        .dropDuplicates() \
                                                            .dropna()


In [0]:
df_cleaned.write.mode("overwrite").saveAsTable(f"{catalog}.silver.silver_customers")